# Diabetic foot-ulcer image classification: worked solution

Worked solution for the exercise in the lesson **[Diabetic foot-ulcer image classification](https://ramkumarmd.com/medical-ai/applied-diabetes/foot-ulcer-image-classification)** (*Applied case studies: diabetes & metabolic health* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](04-foot-ulcer-image-classification.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** AI models can detect diabetic foot ulcers in photographs and measure their size, which could support remote monitoring and earlier referral. But image models trained mostly on lighter skin can perform worse on darker skin, as dermatology AI has shown, so performance must be reported by skin tone. A practical use of wound measurement is tracking healing: a reduction in ulcer area of about half within four weeks is widely used as a sign of progress, and slower healing should prompt review of treatment.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A skin-tone performance gap

Synthetic wound photos, a small CNN, and accuracy reported separately by skin tone.

In [ ]:
import numpy as np
import torch
from torch import nn

def wound_photos(n, seed, dark_share):
    """Synthetic 28x28 'foot photos'. Skin tone sets the overall brightness; an ulcer is a darker,
    textured patch. dark_share: fraction of photos with darker skin tones."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:28, :28]
    dark = rng.random(n) < dark_share
    tone = np.where(dark, rng.uniform(-1.6, -0.8, n), rng.uniform(0.8, 1.6, n))
    images = tone[:, None, None] + rng.normal(0, 0.6, (n, 28, 28))
    labels = rng.integers(0, 2, n)
    for i in np.where(labels == 1)[0]:
        cy, cx, r = rng.integers(7, 21), rng.integers(7, 21), rng.uniform(3, 5)
        patch = (yy - cy) ** 2 + (xx - cx) ** 2 < r ** 2
        images[i][patch] -= 1.2 + rng.normal(0, 0.4, patch.sum())
    to_t = lambda a: torch.tensor(a, dtype=torch.float32)
    return to_t(images).unsqueeze(1), torch.tensor(labels), dark

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(16 * 7 * 7, 2))
    def forward(self, x):
        return self.net(x)

def train(X, y, epochs=8):
    torch.manual_seed(0)
    model = SmallCNN()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        for idx in torch.randperm(len(X)).split(64):
            opt.zero_grad()
            nn.functional.cross_entropy(model(X[idx]), y[idx]).backward()
            opt.step()
    return model

def accuracy_by_tone(model, X, y, dark):
    with torch.no_grad():
        correct = (model(X).argmax(1) == y).numpy()
    return correct[~dark].mean(), correct[dark].mean()

X_tr, y_tr, dark_tr = wound_photos(4000, seed=1, dark_share=0.05)       # 95% lighter skin in training
X_te, y_te, dark_te = wound_photos(2000, seed=2, dark_share=0.5)        # balanced test set
model = train(X_tr, y_tr)
light_acc, dark_acc = accuracy_by_tone(model, X_te, y_te, dark_te)
print(f"Trained on 5% darker skin:  lighter skin {light_acc:.1%}, darker skin {dark_acc:.1%}")

# Trained on 5% darker skin:  lighter skin 99.7%, darker skin 71.6%

### Two mitigations

Continuing from the previous block: retrain on diverse data, or normalise each image's brightness.

In [ ]:
X_bal, y_bal, _ = wound_photos(4000, seed=3, dark_share=0.5)            # diverse training set
light_acc, dark_acc = accuracy_by_tone(train(X_bal, y_bal), X_te, y_te, dark_te)
print(f"Trained on 50% darker skin: lighter skin {light_acc:.1%}, darker skin {dark_acc:.1%}")

def normalise(X):                                                         # per-image standardisation
    return (X - X.mean(dim=(2, 3), keepdim=True)) / X.std(dim=(2, 3), keepdim=True)
light_acc, dark_acc = accuracy_by_tone(train(normalise(X_tr), y_tr), normalise(X_te), y_te, dark_te)
print(f"5% darker skin, normalised: lighter skin {light_acc:.1%}, darker skin {dark_acc:.1%}")

# Trained on 50% darker skin: lighter skin 99.2%, darker skin 97.6%
# 5% darker skin, normalised: lighter skin 99.9%, darker skin 100.0%

### Tracking healing with wound area

Percent area reduction after four weeks for three synthetic patients.

In [ ]:
# Wound area (cm2) measured weekly from segmented photos for three synthetic patients
areas = {"patient A": [4.0, 3.1, 2.4, 1.7, 1.1], "patient B": [3.5, 3.4, 3.3, 3.0, 2.9],
         "patient C": [6.2, 5.6, 4.0, 3.5, 3.0]}
for name, a in areas.items():
    reduction = 1 - a[4] / a[0]                        # percent area reduction after 4 weeks
    flag = "on track" if reduction >= 0.5 else "REVIEW: under 50% reduction at 4 weeks"
    print(f"{name}: {reduction:.0%} smaller after 4 weeks -> {flag}")

# patient A: 72% smaller after 4 weeks -> on track
# patient B: 17% smaller after 4 weeks -> REVIEW: under 50% reduction at 4 weeks
# patient C: 52% smaller after 4 weeks -> on track

## Exercise

**Goal:** Build and audit a foot-ulcer image classifier across skin tones, and design a healing-monitoring rule.

**Data:** Synthetic wound photos in the notebook; optional DFUC datasets under their data-use terms (Generated in the notebook; external datasets under their own licences)

### Step 1

Reproduce the skin-tone experiment and plot accuracy against the share of darker-skin photos in training (5%, 15%, 30%, 50%).

In [ ]:
import matplotlib.pyplot as plt

shares, light, dark_ = [0.05, 0.15, 0.30, 0.50], [], []
for share in shares:
    X, y, _ = wound_photos(4000, seed=10 + int(share * 100), dark_share=share)
    l, d = accuracy_by_tone(train(X, y), X_te, y_te, dark_te)
    light.append(l); dark_.append(d)
    print(f"{share:>4.0%} darker skin in training: lighter {l:.1%}, darker {d:.1%}")
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot([100 * s for s in shares], light, "o-", label="lighter skin")
ax.plot([100 * s for s in shares], dark_, "o-", label="darker skin")
ax.set(xlabel="Darker-skin photos in training (%)", ylabel="Test accuracy", ylim=(0.5, 1.01)); ax.legend()
plt.show()

With this training sample, even 5% darker-skin photos gave 92% accuracy on darker skin, against 71.6% in the lesson's
run with a different random sample. Performance on an under-represented group is unstable as well as lower: it depends
on which few examples happened to be included. More examples make it both higher and more predictable.

### Step 2

Make the ulcer contrast lower on darker skin, a more realistic difference, and check whether normalisation still closes the gap.

In [ ]:
def wound_photos_v2(n, seed, dark_share, dark_contrast=0.5):
    """As wound_photos, but ulcers on darker skin have lower contrast (scaled by dark_contrast)."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:28, :28]
    dark = rng.random(n) < dark_share
    tone = np.where(dark, rng.uniform(-1.6, -0.8, n), rng.uniform(0.8, 1.6, n))
    images = tone[:, None, None] + rng.normal(0, 0.6, (n, 28, 28))
    labels = rng.integers(0, 2, n)
    for i in np.where(labels == 1)[0]:
        cy, cx, r = rng.integers(7, 21), rng.integers(7, 21), rng.uniform(3, 5)
        patch = (yy - cy) ** 2 + (xx - cx) ** 2 < r ** 2
        depth = 1.2 * (dark_contrast if dark[i] else 1.0)
        images[i][patch] -= depth + rng.normal(0, 0.4 * (dark_contrast if dark[i] else 1.0), patch.sum())
    return torch.tensor(images, dtype=torch.float32).unsqueeze(1), torch.tensor(labels), dark

X2_tr, y2_tr, _ = wound_photos_v2(4000, seed=21, dark_share=0.05)
X2_te, y2_te, dark2_te = wound_photos_v2(2000, seed=22, dark_share=0.5)
for label, Xa, Xb in [("raw images", X2_tr, X2_te), ("normalised", normalise(X2_tr), normalise(X2_te))]:
    l, d = accuracy_by_tone(train(Xa, y2_tr), Xb, y2_te, dark2_te)
    print(f"Lower ulcer contrast on darker skin, {label:<11}: lighter {l:.1%}, darker {d:.1%}")
X2_bal, y2_bal, _ = wound_photos_v2(4000, seed=23, dark_share=0.5)
l, d = accuracy_by_tone(train(normalise(X2_bal), y2_bal), normalise(X2_te), y2_te, dark2_te)
print(f"Lower contrast, normalised AND 50% darker skin in training: lighter {l:.1%}, darker {d:.1%}")

When the only difference was overall brightness, normalisation removed it completely. When ulcers really are harder
to see on darker skin (lower contrast), normalisation cannot restore information that is not in the image as strongly;
a gap remains, and diverse training data help more. In real photography, lighting, camera exposure and the way
erythema shows on different skin tones all contribute, so performance must be measured, not assumed.

### Step 3

Report sensitivity and specificity by skin tone with confidence intervals, and decide what minimum performance you would require in each group.

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n
    c, h = (p + z**2 / (2 * n)) / (1 + z**2 / n), z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return c - h, c + h

model2 = train(normalise(X2_tr), y2_tr)
with torch.no_grad():
    pred = model2(normalise(X2_te)).argmax(1).numpy()
truth = y2_te.numpy()
for label, m in [("lighter skin", ~dark2_te), ("darker skin", dark2_te)]:
    tp = np.sum((pred == 1) & (truth == 1) & m); fn = np.sum((pred == 0) & (truth == 1) & m)
    tn = np.sum((pred == 0) & (truth == 0) & m); fp = np.sum((pred == 1) & (truth == 0) & m)
    (sl, sh), (pl, ph) = wilson(tp, tp + fn), wilson(tn, tn + fp)
    print(f"{label:<13} sensitivity {tp / (tp + fn):.1%} ({sl:.1%}-{sh:.1%}), specificity {tn / (tn + fp):.1%} ({pl:.1%}-{ph:.1%})")

A minimum standard should be set in advance for every group, not only overall: for example, sensitivity for ulcers of
at least 90% with the lower confidence limit above 85% in each skin-tone group, and no group more than 5 points
below the best. A model that fails in one group fails the standard, however good the average is; the fix is better
data for that group, not a lower bar.

### Step 4

Extend the healing rule to flag ulcers that grow between visits, and write the message a clinician would see.

In [ ]:
def healing_check(areas_cm2, weeks):
    """Flag growth between consecutive visits, and slow healing at 4 weeks (under 50% area reduction)."""
    messages = []
    for (w0, a0), (w1, a1) in zip(zip(weeks, areas_cm2), zip(weeks[1:], areas_cm2[1:])):
        if a1 > a0 * 1.10:                                   # grown by more than 10%
            messages.append(f"Week {w1}: ulcer area increased from {a0:.1f} to {a1:.1f} cm2 (+{a1 / a0 - 1:.0%}) "
                            f"since week {w0}. Review today for infection, ischaemia or offloading problems.")
    if 4 in weeks and areas_cm2[weeks.index(4)] > 0.5 * areas_cm2[0]:
        change = 1 - areas_cm2[weeks.index(4)] / areas_cm2[0]
        trend = f"reduced by only {change:.0%}" if change >= 0 else f"{-change:.0%} LARGER than at week 0"
        messages.append(f"Week 4: area {trend} (target: at least 50% smaller). Consider escalating care.")
    return messages or ["Healing on track."]

cases = {"patient A": [4.0, 3.1, 2.4, 1.7, 1.1], "patient B": [3.5, 3.4, 3.3, 3.0, 2.9],
         "patient C": [6.2, 5.6, 4.0, 3.5, 3.0], "patient D": [2.0, 1.8, 2.4, 2.6, 2.5]}
for name, a in cases.items():
    print(name)
    for msg in healing_check(a, weeks=[0, 1, 2, 3, 4]):
        print("   ", msg)

The message a clinician sees should state the measurement, the change and the time, and suggest what to check, while
leaving the decision to them. Because the area comes from automatic segmentation, the message should also link to the
photographs, so a clinician can see whether the "growth" is real or a change in camera angle or lighting.

---

Lesson: [https://ramkumarmd.com/medical-ai/applied-diabetes/foot-ulcer-image-classification](https://ramkumarmd.com/medical-ai/applied-diabetes/foot-ulcer-image-classification). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.